In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import os
from flask import Flask, request, render_template, redirect, url_for
from werkzeug.utils import secure_filename
from PIL import Image
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

dataset_path = '/content/drive/MyDrive/archive (3)/teeth_dataset'
train_dir = os.path.join(dataset_path, '/content/drive/MyDrive/archive (3)/teeth_dataset/Trianing')
test_dir = os.path.join(dataset_path, '/content/drive/MyDrive/archive (3)/teeth_dataset/test')

# Load Dataset
datagen = keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

test_generator = datagen.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

# Define CNN Model
model = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train Model
model.fit(train_generator, validation_data=test_generator, epochs=10)

# Save Model
model.save('teeth_model.h5')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 256 images belonging to 2 classes.
Found 32 images belonging to 2 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 31s 4s/step - accuracy: 0.6396 - loss: 0.4840 - val_accuracy: 0.8750 - val_loss: 0.5657
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.9357 - loss: 0.2563 - val_accuracy: 0.8750 - val_loss: 0.9121
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - accuracy: 0.9404 - loss: 0.3413 - val_accuracy: 0.8750 - val_loss: 0.3097
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.9335 - loss: 0.1691 - val_accuracy: 0.8750 - val_loss: 0.1477
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.9550 - loss: 0.1009 - val_accuracy: 1.0000 - val_loss: 0.1689
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.9703 - loss: 0.1288 - val_accuracy: 0.9375 - val_loss: 0.1095
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.9867 - loss: 0.0740 - val_accuracy: 0.8750 - val_loss: 0.2356
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.9698 - loss: 0.2041 - val_accuracy: 0.8438 - val_loss: 0.2377
Epoch 9/

In [ ]:
# prompt: calculate performance metrics

# Evaluate the model
loss, accuracy = model.evaluate(test_generator)

print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

from sklearn.metrics import classification_report, confusion_matrix

# Get predictions from the model
predictions = model.predict(test_generator)
predicted_classes = (predictions > 0.5).astype(int)

# Get true labels
true_classes = test_generator.classes

# Calculate and print the classification report
print(classification_report(true_classes, predicted_classes))

# Calculate and print the confusion matrix
print(confusion_matrix(true_classes, predicted_classes))


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 926ms/step - accuracy: 1.0000 - loss: 0.0492
Test Loss: 0.04915326461195946
Test Accuracy: 1.0
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
              precision    recall  f1-score   support

           0       0.86      0.86      0.86        28
           1       0.00      0.00      0.00         4

    accuracy                           0.75        32
   macro avg       0.43      0.43      0.43        32
weighted avg       0.75      0.75      0.75        32

[[24  4]
 [ 4  0]]


In [ ]:
# prompt: pridection

import numpy as np
from tensorflow import keras

# Load the saved model
model = keras.models.load_model('teeth_model.h5')

# Assuming you have a new image loaded as a numpy array called 'new_image'
# with shape (128, 128, 3) and normalized pixel values (0-1).

# Example: Load and preprocess a new image (replace with your image loading)
from PIL import Image
img = Image.open('/content/drive/MyDrive/archive (3)/teeth_dataset/Trianing/caries/0.jpg').resize((128,128))
# Convert the image to RGB format
img = img.convert('RGB') # This line converts the image to RGB
img_array = np.array(img) / 255.0
new_image = np.expand_dims(img_array, axis=0)

# Make the prediction
prediction = model.predict(new_image)

# Interpret the prediction
predicted_class = (prediction > 0.5).astype(int)[0][0]

if predicted_class == 1:
    print("Prediction: healthy")
else:
    print("Prediction: caries")

print(f"Confidence: {prediction[0][0]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
Prediction: caries
Confidence: 7.018689150923088e-14
